# MDAV — AIForge dataset generation (2x GPU)
FLUX-Fill forgery generator across **both** Kaggle T4s in parallel.

**Before running:** GPU T4x2 + Internet ON; add `HF_TOKEN` secret; accept the FLUX.1-Fill-dev HF license; attach `mdav-code` + `mdav-datasets`.


## 1. Check GPUs + mounts


In [ ]:
import subprocess, os
print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout or 'NO GPU')
for d,_,fs in os.walk('/kaggle/input'):
    if fs: print(d,'->',fs[:3])


## 2. Copy the code in


In [ ]:
%%bash
CODE_DIR=$(dirname "$(find /kaggle/input -name setup_kaggle.py | head -1)")
echo "Code at: $CODE_DIR"; cp -r "$CODE_DIR"/* /kaggle/working/; ls /kaggle/working/


## 3. Environment
`FLUX_NO_OFFLOAD=1` keeps weights on the GPU -> much less host RAM (both shards fit) and faster.
`PYTHONUNBUFFERED=1` makes shard logs show live.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['MDAV_OCR_VERIFY'] = 'changed'
os.environ['FLUX_NUM_INFERENCE_STEPS'] = '20'
os.environ['FLUX_NO_OFFLOAD'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'
print('env:', {k:os.environ[k] for k in ['MDAV_OCR_VERIFY','FLUX_NUM_INFERENCE_STEPS','FLUX_NO_OFFLOAD']})


## 4. Install + cache FLUX (first time ~15-25 min)


In [ ]:
%cd /kaggle/working
!pip install -q -r requirements.txt
!python setup_kaggle.py


## 5. Wire datasets + VERIFY structure


In [ ]:
%%bash
cd /kaggle/working; rm -rf datasets && mkdir -p datasets
DATA_DIR=$(dirname "$(find /kaggle/input -type d -name CORD | head -1)")
echo "Data at: $DATA_DIR"; cp -r "$DATA_DIR"/* datasets/
echo '--- structure ---'; find datasets -maxdepth 2 -type d | sort


## 6. Launch BOTH GPUs in parallel
python -u = unbuffered logs. shard0->GPU0, shard1->GPU1.


In [ ]:
import subprocess, os
def launch(gpu, idx, out):
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)}
    return subprocess.Popen(
        ['python','-u','main.py','--datasets','CORD','--variants-per-doc','2',
         '--shard-count','2','--shard-index',str(idx),'--output-dir',out],
        cwd='/kaggle/working', env=env,
        stdout=open(f'/kaggle/working/shard{idx}.log','w'), stderr=subprocess.STDOUT)
p0 = launch(0,0,'/kaggle/working/mdav_shard_0')
p1 = launch(1,1,'/kaggle/working/mdav_shard_1')
print('launched PIDs:', p0.pid, p1.pid)


## 7. Monitor (re-run anytime)
Both GPUs ~13 GB + high util. poll() None = running; a number = it exited (read its log).


In [ ]:
print('shard0 alive?', p0.poll() is None, '| shard1 alive?', p1.poll() is None)
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used,memory.total --format=csv
print('--- shard0 ---'); !tail -6 /kaggle/working/shard0.log
print('--- shard1 ---'); !tail -6 /kaggle/working/shard1.log


## 8. Inspect + PROVE the mask matches the tamper
Overlays the mask on the forged image in red - the red box must sit on the edited field.


In [ ]:
import glob, os, numpy as np
from PIL import Image
from IPython.display import display
imgs = sorted(glob.glob('/kaggle/working/mdav_shard_*/images/*.png'))
print('accepted samples:', len(imgs))
if imgs:
    p = imgs[-1]; print(p)
    img = Image.open(p).convert('RGB')
    mask = Image.open(p.replace('/images/','/masks/')).convert('L')
    a = np.array(img); a[np.array(mask) > 127] = [255,0,0]
    display(Image.fromarray(a))


## 9. Merge shard outputs (when done)


In [ ]:
%%bash
cd /kaggle/working; mkdir -p outputs/{images,masks,annotations}
cp -n mdav_shard_*/images/* outputs/images/ 2>/dev/null || true
cp -n mdav_shard_*/masks/* outputs/masks/ 2>/dev/null || true
cp -n mdav_shard_*/annotations/* outputs/annotations/ 2>/dev/null || true
echo "merged images: $(ls outputs/images 2>/dev/null | wc -l)"
